# Bank NLP — End-to-End Customer Feedback Analysis

## Business Case

Banks receive large amounts of unstructured text from:

- customer complaints,
- call center notes,
- email,
- app reviews,
- survey comments,
- social media,
- service requests.

NLP converts this text into structured information that can support business decisions.

This notebook builds an end-to-end banking NLP pipeline covering:

1. Text preprocessing
2. Exploratory text analysis
3. Sentiment classification
4. TF-IDF
5. Logistic Regression
6. Evaluation
7. Topic analysis with NMF
8. Keyword extraction
9. Prediction on new feedback
10. Business insights
11. Production architecture

> Synthetic data for education only.

## 1. What is NLP?

**Natural Language Processing (NLP)** is a field of machine learning and AI for working with human language.

For banking:

```text
Customer Feedback
        ↓
       NLP
        ↓
Structured Information
        ↓
Sentiment / Topic / Intent
        ↓
Business Action
```

Example:

> "Aplikasi mobile banking sering error."

Can become:

```text
Sentiment = Negative
Topic = Mobile Banking
Priority = Potential Service Issue
```

## 2. Banking NLP Use Cases

### Customer Feedback
Detect positive, negative, and neutral sentiment.

### Complaint Classification
Automatically classify complaints by topic.

### Call Center Analytics
Summarize large volumes of conversations.

### Intent Detection
Identify what customers want.

Examples:
- check balance,
- reset PIN,
- apply for loan,
- report failed transaction.

### Fraud / Risk
Analyze text-based reports and investigation notes.

### Voice of Customer
Find recurring issues and emerging themes.

## 3. Dataset

The synthetic customer-feedback dataset — Indonesian mobile-banking comments with a sentiment label — is loaded into a DataFrame. Each row is one piece of feedback text that the notebook will clean, classify and turn into topics.


In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support
)
from sklearn.decomposition import NMF

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns",100)

df=pd.read_csv("bank_customer_feedback_sample.csv")
print("Rows:",len(df))
display(df.head())

## 4. Dataset Fields

| Field | Description |
|---|---|
| Feedback_ID | Feedback identifier |
| Text | Customer text |
| Topic | Synthetic business topic |
| Channel | Feedback channel |
| Sentiment | Target label |

For sentiment modeling, **Text** is the input and **Sentiment** is the target.

`Topic` and `Channel` are retained for business analysis but are not used as model inputs in the basic sentiment model.

## 5. Data Quality

Missing texts, duplicate feedback and label consistency are checked. Duplicate or empty comments would distort both the sentiment classifier and the topic model built later.


In [ ]:
display(df.isna().sum().to_frame("Missing"))
print("Duplicate texts:",df["Text"].duplicated().sum())
display(df["Sentiment"].value_counts().to_frame("Count"))

## 6. Sentiment Distribution

The class balance between positive, negative and neutral feedback is visualised. Imbalanced classes inflate naive accuracy, so this view drives the choice of metrics and baseline.


In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=df,x="Sentiment")
plt.title("Customer Sentiment Distribution")
plt.show()

display(pd.crosstab(df["Channel"],df["Sentiment"],normalize="index").round(3))

## 7. Text Preprocessing

Typical preprocessing steps:

1. Lowercase
2. Remove URLs
3. Remove punctuation
4. Normalize whitespace

We avoid blindly removing every stopword because some words can carry meaning depending on the task.

For Indonesian banking text, preprocessing should also consider:
- slang,
- abbreviations,
- bank/product names,
- Indonesian stopwords,
- spelling variations.

In [ ]:
def clean_text(text):
    text=str(text).lower()
    text=re.sub(r"http\S+|www\.\S+"," ",text)
    text=re.sub(r"[^a-zA-ZÀ-ÿ0-9\s]"," ",text)
    text=re.sub(r"\s+"," ",text).strip()
    return text

df["Clean_Text"]=df["Text"].apply(clean_text)

display(df[["Text","Clean_Text"]].head(10))

## 8. Train-Test Split

We use stratification so the class distribution is approximately preserved.

```text
80% Train
20% Test
```

The test set remains unseen during model training.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(
    df["Clean_Text"],
    df["Sentiment"],
    test_size=.20,
    stratify=df["Sentiment"],
    random_state=42
)

print("Train:",len(X_train))
print("Test:",len(X_test))

## 9. TF-IDF

TF-IDF represents text numerically.

It gives higher weight to words that are:
- frequent in a document,
- but relatively less common across all documents.

```text
Text
 ↓
TF-IDF
 ↓
Numerical Feature Matrix
 ↓
Machine Learning Model
```

In [ ]:
vectorizer=TfidfVectorizer(
    ngram_range=(1,2),
    min_df=2,
    max_df=.95,
    sublinear_tf=True
)

X_train_tfidf=vectorizer.fit_transform(X_train)
X_test_tfidf=vectorizer.transform(X_test)

print("Train matrix:",X_train_tfidf.shape)
print("Test matrix:",X_test_tfidf.shape)

## 10. Logistic Regression

For text classification, Logistic Regression is a strong and interpretable baseline.

Pipeline:

```text
Text
 ↓
TF-IDF
 ↓
Logistic Regression
 ↓
Sentiment
```

In [ ]:
model=LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_tfidf,y_train)
pred=model.predict(X_test_tfidf)

print(classification_report(y_test,pred))

## 11. Evaluation

Accuracy, precision, recall and F1 are computed on the held-out test set. Per-class metrics matter here because misclassifying a complaint as positive hides a service problem from the bank.


In [ ]:
print("Accuracy:",round(accuracy_score(y_test,pred),4))

precision,recall,f1,_=precision_recall_fscore_support(
    y_test,pred,average="weighted"
)

print("Weighted Precision:",round(precision,4))
print("Weighted Recall:",round(recall,4))
print("Weighted F1:",round(f1,4))

In [ ]:
cm=confusion_matrix(y_test,pred,labels=model.classes_)
sns.heatmap(
    cm,annot=True,fmt="d",cmap="Blues",
    xticklabels=model.classes_,
    yticklabels=model.classes_
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Sentiment Confusion Matrix")
plt.show()

## 12. Build Production-Style Pipeline

Instead of manually transforming data, combine TF-IDF and the classifier into one pipeline.

Benefits:
- fewer preprocessing mistakes,
- easier deployment,
- same transformation during training and prediction.

In [ ]:
sentiment_pipeline=Pipeline([
    ("tfidf",TfidfVectorizer(
        ngram_range=(1,2),
        min_df=2,
        max_df=.95,
        sublinear_tf=True
    )),
    ("classifier",LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

sentiment_pipeline.fit(X_train,y_train)
print("Pipeline trained.")

## 13. Predict New Customer Feedback

The fitted pipeline scores brand-new feedback sentences never seen during training, demonstrating exactly how the bank would classify incoming comments in production.


In [ ]:
new_feedback=[
    "Aplikasi mobile banking saya sangat mudah digunakan",
    "Transaksi saya gagal dan saldo belum kembali",
    "Bagaimana cara mengganti PIN kartu saya",
    "Customer service sangat cepat membantu"
]

new_pred=sentiment_pipeline.predict(new_feedback)
new_prob=sentiment_pipeline.predict_proba(new_feedback).max(axis=1)

result=pd.DataFrame({
    "Text":new_feedback,
    "Predicted_Sentiment":new_pred,
    "Confidence":new_prob
})

display(result)

## 14. Confidence Interpretation

The probability produced by a classifier is a model confidence score.

It should **not automatically be interpreted as a true probability of customer sentiment** without calibration and validation.

For operational use, define an escalation rule such as:

```text
Negative + high confidence → prioritize
Low confidence → human review
```

## 15. Most Important TF-IDF Features

The largest logistic-regression coefficients per sentiment class are listed together with their TF-IDF terms. This shows which words push feedback toward negative or positive — both a sanity check and a source of insight.


In [ ]:
feature_names=np.array(vectorizer.get_feature_names_out())

for cls in model.classes_:
    coef=model.coef_[list(model.classes_).index(cls)]
    top=feature_names[np.argsort(coef)[-15:]][::-1]
    print(f"\nTop terms associated with {cls}:")
    print(", ".join(top))

## 16. Topic Modeling with NMF

Sentiment answers:

> "Is the customer positive or negative?"

Topic modeling answers:

> "What themes are customers talking about?"

NMF decomposes the document-term matrix into latent topics.

In [ ]:
topic_vectorizer=TfidfVectorizer(
    stop_words=None,
    ngram_range=(1,2),
    min_df=2,
    max_df=.95
)

X_topic=topic_vectorizer.fit_transform(df["Clean_Text"])

n_topics=6
nmf=NMF(
    n_components=n_topics,
    random_state=42,
    init="nndsvda",
    max_iter=500
)

W=nmf.fit_transform(X_topic)
H=nmf.components_

topic_terms=np.array(topic_vectorizer.get_feature_names_out())

for i,topic in enumerate(H):
    top_terms=topic_terms[topic.argsort()[-10:][::-1]]
    print(f"Topic {i+1}: {', '.join(top_terms)}")

## 17. Assign Dominant Topic

Each feedback item is assigned its strongest NMF topic and the corresponding topic weight. This turns unsupervised topics into a per-comment tag the business can count and track.


In [ ]:
df["Topic_Model"]=W.argmax(axis=1)+1
df["Topic_Score"]=W.max(axis=1)

display(df[["Text","Topic_Model","Topic_Score","Sentiment"]].head(15))

## 18. Topic Distribution by Sentiment

A cross-tabulation of topics versus sentiment reveals which subjects drive complaints and which drive praise, pointing product and service teams to the areas where improvement pays off most.


In [ ]:
topic_sentiment=pd.crosstab(
    df["Topic_Model"],
    df["Sentiment"],
    normalize="index"
)

display(topic_sentiment.round(3))

topic_sentiment.plot(kind="bar",stacked=True,figsize=(10,5))
plt.title("Topic Composition by Sentiment")
plt.xlabel("Topic")
plt.ylabel("Proportion")
plt.show()

## 19. Keyword Frequency

A simple keyword frequency analysis can provide a fast view of recurring customer language.

For production Indonesian NLP, consider a curated stopword list and domain-specific normalization.

In [ ]:
from collections import Counter

words=[]
for text in df["Clean_Text"]:
    words.extend(text.split())

word_counts=pd.DataFrame(
    Counter(words).most_common(20),
    columns=["Word","Count"]
)

display(word_counts)

plt.figure(figsize=(10,6))
sns.barplot(data=word_counts.head(15),x="Count",y="Word")
plt.title("Most Frequent Words")
plt.show()

## 20. Complaint Analytics

For banking operations, negative sentiment is often the first filter.

Example:

```text
All Feedback
     ↓
Sentiment
     ↓
Negative
     ↓
Topic
     ↓
Prioritize by business impact
```

This can support service recovery and root-cause analysis.

In [ ]:
negative=df[df["Sentiment"]=="negative"].copy()

display(
    negative.groupby(["Channel","Topic"])
    .size()
    .reset_index(name="Complaints")
    .sort_values("Complaints",ascending=False)
    .head(20)
)

## 21. Simple Complaint Priority

A production system can combine NLP output with business metadata.

Example:

```text
Priority =
    Sentiment
  + Topic
  + Customer Value
  + Transaction Impact
  + Recency
```

The NLP model provides one signal; operational priority should not rely on sentiment alone.

In [ ]:
# Example rule-based priority demonstration
priority_map={"negative":3,"neutral":1,"positive":0}

df["Sentiment_Priority"]=df["Sentiment"].map(priority_map)

df["Priority"] = np.where(
    df["Sentiment"]=="negative","High",
    np.where(df["Sentiment"]=="neutral","Medium","Low")
)

display(df[["Text","Sentiment","Topic","Channel","Priority"]].head(15))

## 22. NLP Pipeline Architecture

```text
Customer Text
     ↓
Text Cleaning
     ↓
Tokenization / Vectorization
     ↓
TF-IDF / Embeddings
     ↓
┌─────────────────────────┐
│ Sentiment Classification │
│ Topic Modeling           │
│ Intent Classification    │
└─────────────────────────┘
     ↓
Business Rules
     ↓
Dashboard / CRM / Alert
```

## 23. Advanced NLP

The TF-IDF + Logistic Regression approach is a strong baseline, but modern systems can use:

### Word Embeddings
- Word2Vec
- FastText

### Transformer Models
- BERT
- IndoBERT
- multilingual transformers

### LLM-based NLP
- summarization
- classification
- extraction
- conversation analysis

For Indonesian banking data, domain-specific language models can be evaluated against simpler baselines.

## 24. Important Banking NLP Considerations

### Privacy

Customer messages can contain:
- account numbers,
- phone numbers,
- identity information,
- transaction information.

Apply masking and access controls.

### Data Leakage

Do not allow future information or manually assigned outcomes to enter training features.

### Human Review

High-impact customer decisions should have appropriate human oversight.

### Monitoring

Track:
- data drift,
- new vocabulary,
- sentiment distribution,
- model performance,
- false positives/negatives.

## 25. Common NLP Mistakes

1. Training and testing after vectorization.
2. Using future information.
3. Removing words without understanding their meaning.
4. Ignoring Indonesian slang and abbreviations.
5. Evaluating only accuracy.
6. Treating model confidence as certainty.
7. Ignoring class imbalance.
8. Deploying without monitoring vocabulary drift.

## 26. Production Deployment

A possible banking architecture:

```text
Mobile App / Call Center / Email / Survey
                 ↓
          Secure Data Layer
                 ↓
          NLP Preprocessing
                 ↓
        NLP Model / API Service
          ↓        ↓        ↓
      Sentiment   Topic    Intent
          └────────┼────────┘
                   ↓
             CRM / Dashboard
                   ↓
          Service Recovery
                   ↓
             Feedback Loop
                   ↓
              Retraining
```

# Final Executive Summary

## Business Problem

Banks receive huge volumes of unstructured customer feedback.

## NLP Solution

Convert text into actionable information:

```text
Customer Feedback
       ↓
      NLP
       ↓
┌──────────────┐
│ Sentiment    │
│ Topic        │
│ Intent       │
└──────────────┘
       ↓
Business Action
```

## Baseline Model

```text
TF-IDF
   ↓
Logistic Regression
   ↓
Sentiment Classification
```

## Additional Analysis

```text
NMF
 ↓
Topic Modeling
```

## Business Value

NLP can support:

- customer experience,
- complaint monitoring,
- service recovery,
- call center analytics,
- product feedback,
- voice-of-customer dashboards.

> **The goal of NLP in banking is not merely to classify text, but to turn customer language into structured information that can support better operational decisions.**